In [3]:
!git clone -b megha-work https://github.com/meghagangal362015/cmpe257-early-reading-prediction.git
%cd cmpe257-early-reading-prediction

Cloning into 'cmpe257-early-reading-prediction'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 47 (delta 6), reused 36 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 830.52 KiB | 3.21 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/cmpe257-early-reading-prediction/cmpe257-early-reading-prediction


In [5]:
!git branch --show-current
!ls

megha-work
'CMPE-257-01_Project-Proposal (2).pdf'	 data   README.md   reports   src


In [6]:
!sed -n '1,240p' src/build_dataset.py

"""Build the combined ECRI dataset (Woodcock + Stanford targets).

Run from the repo root:
    python src/build_dataset.py

Reads the 3 original CSVs from data/ and writes data/processed/ecri_combined.csv.

What this script does:
  - one row per student, fall (start-of-year) features only
  - joins school-level info from MetaData.csv
  - adds MID-YEAR (winter) fluency scores as *_mid columns, for the mid-year scenario only
  - adds spring scores and gains, used ONLY to build targets
  - adds the Stanford growth target (gain >= median), like the paper's Woodcock targets
  - keeps students with at least one target
  - adds a fixed train/validation/test split (70/15/15) that keeps whole schools together

What it does NOT do (do these inside cross-validation to avoid leakage):
  - fill missing values, scale features, or select features
"""
import pandas as pd
import numpy as np

RAW_DIR = "data"
OUT = "data/processed/ecri_combined.csv"

df = pd.read_csv(f"{RAW_DIR}/ECRIDataSet.csv")
raw = 

In [7]:
import pandas as pd

data = pd.read_csv("data/processed/ecri_combined.csv")

print("Dataset shape:", data.shape)
print("Columns:")
print(data.columns.tolist())

print("\nWord Identification labels:")
print(data["RMwidRS_target"].notna().sum())

print("\nWord Attack labels:")
print(data["RMwdaRS_target"].notna().sum())

Dataset shape: (5679, 46)
Columns:
['StuID', 'Gender', 'Tier', 'SchlID', 'TeachID', 'LEP', 'SpEd', 'Tx', 'Cohort', 'Age1b', 'Tier2_N', 'grp_rate', 'rcmistot', 'gnrl_fid', 'TKPctCrct', 'Tier2MinRd', 'TYearsTeaching', 'NWFcls', 'NWFwrc', 'ORFwc', 'SAwrS', 'SAsrS', 'SAtoS', 'RMwidRS', 'RMwdaRS', 'RMwidRS_target', 'RMwdaRS_target', 'Schl_FreeLunch', 'Schl_AmericanIndian', 'Schl_Asian', 'Schl_Hispanic', 'Schl_AfricanAmerican', 'Schl_White', 'Schl_NativeHawaiian', 'Schl_MoreRaces', 'NWFcls_mid', 'NWFwrc_mid', 'ORFwc_mid', 'SAtoS_spring', 'RMwidRS_spring', 'RMwdaRS_spring', 'SAtoS_gain', 'RMwidRS_gain', 'RMwdaRS_gain', 'SAtoS_target', 'split']

Word Identification labels:
946

Word Attack labels:
946


Step 1: Load the prepared splits

In [8]:
import pandas as pd

train = pd.read_csv("data/processed/train.csv")
validation = pd.read_csv("data/processed/validation.csv")
test = pd.read_csv("data/processed/test.csv")

print(train.shape)
print(validation.shape)
print(test.shape)

(3993, 46)
(843, 46)
(843, 46)


Step 2: Add the preprocessing code

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

drop_columns = [
    "StuID",
    "SchlID",
    "TeachID",
    "split",
    "RMwidRS_target",
    "RMwdaRS_target",
    "SAtoS_target",
    "grp_rate",
    "rcmistot",
    "gnrl_fid",
    "Tier2MinRd"
]

def clean_features(df):
    columns_to_drop = []

    for column in df.columns:
        if (
            column in drop_columns
            or "_spring" in column
            or "_gain" in column
            or "_mid" in column
        ):
            columns_to_drop.append(column)

    return df.drop(columns=columns_to_drop, errors="ignore")

Step 3: Prepare one Woodcock target

In [15]:
target = "RMwidRS_target"

train_target = train.dropna(subset=[target])
validation_target = validation.dropna(subset=[target])
test_target = test.dropna(subset=[target])

X_train = clean_features(train_target.drop(columns=[target]))
X_val = clean_features(validation_target.drop(columns=[target]))
X_test = clean_features(test_target.drop(columns=[target]))

y_train = train_target[target].astype(int)
y_val = validation_target[target].astype(int)
y_test = test_target[target].astype(int)

numeric_columns = X_train.select_dtypes(include=["number"]).columns
categorical_columns = X_train.select_dtypes(exclude=["number"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_columns
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (700, 26) (700,)
Validation: (141, 26) (141,)
Testing: (105, 26) (105,)


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_columns = X_train.select_dtypes(include=["number"]).columns
categorical_columns = X_train.select_dtypes(exclude=["number"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_columns
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_columns
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_val_processed.shape)
print(X_test_processed.shape)

(700, 26)
(141, 26)
(105, 26)


In [18]:
target_attack = "RMwdaRS_target"

train_attack = train.dropna(subset=[target_attack])
validation_attack = validation.dropna(subset=[target_attack])
test_attack = test.dropna(subset=[target_attack])

X_train_attack = clean_features(
    train_attack.drop(columns=[target_attack])
)
X_val_attack = clean_features(
    validation_attack.drop(columns=[target_attack])
)
X_test_attack = clean_features(
    test_attack.drop(columns=[target_attack])
)

y_train_attack = train_attack[target_attack].astype(int)
y_val_attack = validation_attack[target_attack].astype(int)
y_test_attack = test_attack[target_attack].astype(int)

preprocessor_attack = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            X_train_attack.select_dtypes(include=["number"]).columns
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            X_train_attack.select_dtypes(exclude=["number"]).columns
        )
    ]
)

X_train_attack_processed = preprocessor_attack.fit_transform(X_train_attack)
X_val_attack_processed = preprocessor_attack.transform(X_val_attack)
X_test_attack_processed = preprocessor_attack.transform(X_test_attack)

print("Training:", X_train_attack.shape, y_train_attack.shape)
print("Validation:", X_val_attack.shape, y_val_attack.shape)
print("Testing:", X_test_attack.shape, y_test_attack.shape)

Training: (700, 26) (700,)
Validation: (141, 26) (141,)
Testing: (105, 26) (105,)
